#  CIFAR-10 Image Classification: ANN vs CNN
### A Production-Grade Deep Learning Study

> **Audience:** DS/ML practitioners with ~2 years experience  
> **Goal:** Build, train, evaluate and deeply analyze ANN vs CNN on CIFAR-10 — from theory to production insights

---

##  What This Notebook Covers

| Section | Topic |
|---|---|
| 1 | Theory: Why images are hard for ANNs |
| 2 | CIFAR-10 EDA & preprocessing |
| 3 | ANN baseline — build, train, evaluate |
| 4 | CNN — spatial feature learning |
| 5 | Regularization strategies: Dropout, BatchNorm |
| 6 | Data Augmentation: the generalization lever |
| 7 | Training dynamics: LR scheduling, Early Stopping |
| 8 | Performance analysis & model comparison |
| 9 | Error analysis: confusion matrix + failure cases |
| 10 | Production takeaways |


---
#  Section 1 — Theory Deep Dive

## 1.1 Why Image Classification is Hard

A 32×32 RGB image = **3,072 raw pixel values**. The challenge isn't just storing them — it's that:

- The **same object** can appear at different positions, scales, rotations, lighting
- Raw pixels have **high intra-class variance** (many ways a "cat" looks)
- Raw pixels have **low inter-class variance** for hard pairs (cat vs dog)
- Pixel space is **not robust** — shifting an image by 1 pixel creates an entirely different input vector

---

## 1.2 The ANN Approach — and Why It Struggles

An ANN flattens the 32×32×3 image into a **3,072-dimensional vector** and passes it through dense layers.

**Problems:**

1. **Spatial invariance destroyed** — pixel at position (5,5) has no relationship to (6,5) in the flattened form
2. **Parameter explosion** — a 512-neuron first layer requires `3072 × 512 = 1.57M` parameters just for one layer
3. **No weight sharing** — the network must learn the same edge/texture feature independently at every spatial location
4. **Overfitting prone** — huge parameter count with small training signal per example
5. **Not translation invariant** — if a dog shifts 2 pixels left, the entire activation pattern changes completely

```
ANN View of an Image:
[p1, p2, p3, ..., p3072]  → Dense(512) → Dense(256) → Dense(10)
   (all spatial context lost)
```

---

## 1.3 The CNN Approach — Spatial Intelligence

CNNs solve the problems above with three key ideas:

###  Key Idea 1: Local Receptive Fields
A 3×3 conv filter looks at **9 pixels at a time** — neighboring pixels that actually have spatial relationship.

###  Key Idea 2: Weight Sharing
The **same filter** slides across the entire image — so an edge detector learned in the top-left corner is automatically applied everywhere. This reduces parameters dramatically.

```
ANN: 3072 × 512 = 1,572,864 params for first layer
CNN: 3×3×3 filter = just 27 params (+ bias), reused across all locations!
```

###  Key Idea 3: Hierarchical Feature Learning
```
Layer 1 Conv → Detects: edges, corners, color gradients
Layer 2 Conv → Detects: textures, patterns, simple shapes  
Layer 3 Conv → Detects: object parts (wheels, ears, wings)
Dense head  → Assembles parts into class decisions
```

###  Key Idea 4: Pooling = Translation Invariance
MaxPooling(2×2) downsamples by taking the max value — if a feature shifts by 1-2 pixels, the same max survives. This gives **approximate translation invariance**.

---

## 1.4 Regularization Theory

### Dropout
- Randomly zeros out `p` fraction of neurons during training
- Forces redundant representations — no single neuron can dominate
- At inference: all neurons active, weights scaled by `(1-p)`
- Approximates **ensemble learning** of 2^N sub-networks

### Batch Normalization
- Normalizes layer inputs to zero mean, unit variance per mini-batch
- Reduces **internal covariate shift** — downstream layers see a stable input distribution
- Allows **higher learning rates** without divergence
- Acts as mild regularizer (due to mini-batch noise in statistics)
- Formula: `x_hat = (x - μ_B) / sqrt(σ²_B + ε)`, then `y = γ*x_hat + β`

### Data Augmentation
- Artificially expands training data by applying label-preserving transforms
- Horizontal flips, rotation, zoom, color jitter = different "views" of same concept
- Forces the model to learn **transformation-invariant** representations
- The single most cost-effective regularization technique for image models

---

## 1.5 CIFAR-10 Dataset Profile

| Property | Value |
|---|---|
| Images | 60,000 (50K train / 10K test) |
| Resolution | 32 × 32 × 3 (RGB) |
| Classes | 10 (balanced, 6000 per class) |
| Challenge | Low resolution + visually similar classes |
| Human accuracy | ~94% |
| SOTA accuracy | ~99.5% (EfficientNet + heavy aug) |

**Hard pairs:** Cat vs Dog, Automobile vs Truck, Bird vs Airplane — these share textures and shapes that confuse both humans and models.

---
#  Section 2 — Setup & Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(f'TensorFlow: {tf.__version__}')
print(f'GPUs available: {len(tf.config.list_physical_devices("GPU"))}')

# Style for all plots
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('darkgrid')

---
#  Section 3 — Data Loading & EDA

In [ ]:
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = tf.keras.datasets.cifar10.load_data()

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print('=== Dataset Profile ===')
print(f'Train:       {x_train_raw.shape}  | dtype: {x_train_raw.dtype}')
print(f'Test:        {x_test_raw.shape}   | dtype: {x_test_raw.dtype}')
print(f'Pixel range: [{x_train_raw.min()}, {x_train_raw.max()}]')
print(f'Classes:     {len(CLASS_NAMES)}')
print(f'\nClass balance (train):')
unique, counts = np.unique(y_train_raw, return_counts=True)
for cls, cnt in zip(CLASS_NAMES, counts):
    print(f'  {cls:<15}: {cnt:,}')

In [ ]:
# Visualize sample images — one per class
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('CIFAR-10: One Sample per Class', fontsize=14, fontweight='bold')

for class_idx, (ax, name) in enumerate(zip(axes.flat, CLASS_NAMES)):
    # Grab first image of each class
    img_idx = np.where(y_train_raw.flatten() == class_idx)[0][0]
    ax.imshow(x_train_raw[img_idx])
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Pixel distribution analysis — this informs normalization strategy
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
channel_names = ['Red', 'Green', 'Blue']
colors = ['#e74c3c', '#2ecc71', '#3498db']

for ch, (ax, cname, color) in enumerate(zip(axes, channel_names, colors)):
    ax.hist(x_train_raw[:1000, :, :, ch].flatten(), bins=50, 
            color=color, alpha=0.7, edgecolor='black', linewidth=0.3)
    ax.set_title(f'{cname} Channel Distribution', fontweight='bold')
    ax.set_xlabel('Pixel Value (0-255)')
    ax.set_ylabel('Frequency')
    mean_val = x_train_raw[:1000, :, :, ch].mean()
    ax.axvline(mean_val, color='black', linestyle='--', label=f'Mean: {mean_val:.1f}')
    ax.legend()

plt.suptitle('Channel-wise Pixel Distribution (1000 sample images)', fontweight='bold')
plt.tight_layout()
plt.show()

print('\n💡 Insight: Pixel values are NOT centered at 0 and have different means per channel.')
print('   Simple /255 normalization → [0,1]. Mean subtraction → zero-centered (better for deep nets).')

---
#  Section 4 — Preprocessing

## Why Normalize?

Raw pixels range 0–255. When we pass these into a network:
- Gradients become **very large** (unstable training)
- Weight initialization (e.g. He/Glorot) assumes **zero-mean, unit-variance** inputs
- Optimizer learning rate needs to compensate → harder to tune

**Two common strategies:**

1. **Min-max scaling:** `x / 255.0` → range [0, 1]. Simple, fast, preserves distribution shape.
2. **Standardization:** `(x - mean) / std` per channel → zero mean, unit std. Better for deeper networks.

We'll use simple `/255` for this study to keep the focus on architecture differences.

In [ ]:
# Normalize
x_train = x_train_raw.astype('float32') / 255.0
x_test  = x_test_raw.astype('float32')  / 255.0

y_train = y_train_raw  # shape: (50000, 1)
y_test  = y_test_raw   # shape: (10000, 1)

# For ANN: flatten spatial dims
x_train_flat = x_train.reshape(len(x_train), -1)  # (50000, 3072)
x_test_flat  = x_test.reshape(len(x_test), -1)    # (10000, 3072)

print('Normalized data:')
print(f'  x_train (CNN input): {x_train.shape} | range [{x_train.min():.2f}, {x_train.max():.2f}]')
print(f'  x_train_flat (ANN input): {x_train_flat.shape}')
print(f'  y_train: {y_train.shape}')

---
#  Section 5 — ANN Baseline

## Architecture Design Rationale

```
Input (3072) → Dense(512, ReLU) → Dropout(0.3)
             → Dense(256, ReLU) → Dropout(0.3)
             → Dense(128, ReLU)
             → Dense(10, Softmax)
```

**Why this architecture:**
- **Funnel shape** (3072 → 512 → 256 → 128 → 10): Progressively compress representation toward class decisions
- **ReLU**: Non-saturating activation — gradients don't vanish like sigmoid/tanh
- **Dropout**: After larger layers (where overfitting risk is highest)
- **Softmax output**: Converts logits to probability distribution over 10 classes

**Parameter count estimate:**
- Layer 1: `3072 × 512 + 512 = ~1.57M`
- Layer 2: `512 × 256 + 256 = ~131K`
- Layer 3: `256 × 128 + 128 = ~32K`
- Layer 4: `128 × 10 + 10 = ~1.3K`
- **Total: ~1.73M parameters**

In [ ]:
def build_ann():
    model = models.Sequential([
        # Block 1
        layers.Dense(512, activation='relu', input_shape=(3072,),
                     kernel_initializer='he_uniform'),  # he_uniform: optimal for ReLU
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        # Block 2
        layers.Dense(256, activation='relu', kernel_initializer='he_uniform'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        # Block 3
        layers.Dense(128, activation='relu', kernel_initializer='he_uniform'),
        layers.Dropout(0.2),
        
        # Output
        layers.Dense(10, activation='softmax')
    ], name='ANN_Baseline')
    return model

ann_model = build_ann()
ann_model.summary()

In [ ]:
# Compile with Adam — adaptive learning rate optimizer
# Adam = momentum + RMSProp: handles sparse gradients, adapts per-parameter LR
ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks: don't train blindly
ann_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,           # Stop if val_loss doesn't improve for 5 epochs
        restore_best_weights=True,  # Revert to best checkpoint automatically
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,           # Halve LR when plateauing
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

print('Training ANN...')
ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=30,
    validation_split=0.1,
    batch_size=128,
    callbacks=ann_callbacks,
    verbose=1
)

In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test, verbose=0)
print(f'\n📊 ANN Results:')
print(f'   Test Accuracy : {ann_test_acc:.4f} ({ann_test_acc*100:.2f}%)')
print(f'   Test Loss     : {ann_test_loss:.4f}')

In [ ]:
def plot_training_curves(history, title, color_train='#3498db', color_val='#e74c3c'):
    """Plot accuracy and loss curves with overfitting gap visualization."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{title} — Training Dynamics', fontsize=13, fontweight='bold')
    
    epochs = range(1, len(history.history['accuracy']) + 1)
    
    # Accuracy
    ax1.plot(epochs, history.history['accuracy'], color=color_train, 
             linewidth=2, label='Train', marker='o', markersize=3)
    ax1.plot(epochs, history.history['val_accuracy'], color=color_val, 
             linewidth=2, label='Validation', marker='s', markersize=3, linestyle='--')
    ax1.fill_between(epochs, history.history['accuracy'], 
                     history.history['val_accuracy'], alpha=0.1, color='gray',
                     label='Overfit gap')
    ax1.set_title('Accuracy', fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.4)
    
    # Loss
    ax2.plot(epochs, history.history['loss'], color=color_train, 
             linewidth=2, label='Train', marker='o', markersize=3)
    ax2.plot(epochs, history.history['val_loss'], color=color_val, 
             linewidth=2, label='Validation', marker='s', markersize=3, linestyle='--')
    ax2.set_title('Loss (Sparse Categorical Crossentropy)', fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.4)
    
    # Annotate best val accuracy
    best_epoch = np.argmax(history.history['val_accuracy'])
    best_acc = history.history['val_accuracy'][best_epoch]
    ax1.axvline(best_epoch + 1, color='green', linestyle=':', linewidth=1.5,
                label=f'Best epoch: {best_epoch+1} ({best_acc:.3f})')
    ax1.legend()
    
    plt.tight_layout()
    plt.show()
    
    overfitting_gap = (
        history.history['accuracy'][-1] - history.history['val_accuracy'][-1]
    )
    print(f'  Best Val Accuracy : {best_acc:.4f} at epoch {best_epoch+1}')
    print(f'  Final Overfit Gap : {overfitting_gap:.4f} (train - val accuracy)')

plot_training_curves(ann_history, 'ANN Baseline')

---
#  Section 6 — CNN Architecture

## Architecture Design — Incremental Complexity

```
Input (32×32×3)
  ↓
Conv2D(32, 3×3) → BN → ReLU → Conv2D(32, 3×3) → BN → ReLU → MaxPool(2×2) → Dropout(0.25)
  ↓ (16×16×32)
Conv2D(64, 3×3) → BN → ReLU → Conv2D(64, 3×3) → BN → ReLU → MaxPool(2×2) → Dropout(0.25)
  ↓ (8×8×64 = 4096)
Conv2D(128, 3×3) → BN → ReLU → GlobalAveragePooling2D
  ↓ (128,)
Dense(256, ReLU) → Dropout(0.5) → Dense(10, Softmax)
```

## Key Design Choices Explained

### Double Conv Blocks
Using two conv layers before pooling (`Conv → Conv → Pool`) is a **VGG-style** pattern. The first conv captures simple features, the second composes them into more complex patterns — before we lose spatial resolution through pooling.

### Increasing Filter Count (32 → 64 → 128)
Early layers detect few feature types (edges, colors) but at high resolution.  
Deeper layers detect many feature combinations at lower resolution.  
Doubling filters compensates for halving spatial resolution — total information capacity stays roughly constant.

### Global Average Pooling (GAP) vs Flatten
- `Flatten(6×6×128)` = 4608 inputs to dense head
- `GlobalAveragePooling()` = 128 values (one per feature map)
- GAP dramatically reduces parameters and acts as **structural regularizer**
- More spatial invariance: not tied to exact feature location

### BatchNorm Placement
Placed **after Conv, before activation** (per original paper).  
Some practitioners prefer after activation — minimal difference in practice.

**Parameter count (estimated):**
- Conv blocks: ~265K params
- Dense head: ~33K params
- **Total: ~298K — 6x fewer than ANN, but far more powerful**

In [ ]:
def build_cnn():
    model = models.Sequential(name='CNN_VGG_Style')
    
    # === Block 1: 32 filters ===
    # Two conv layers before pooling → richer feature composition
    model.add(layers.Conv2D(32, (3,3), padding='same', input_shape=(32,32,3)))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    
    model.add(layers.Conv2D(32, (3,3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    
    model.add(layers.MaxPooling2D((2,2)))  # 32×32 → 16×16
    model.add(layers.Dropout(0.25))
    
    # === Block 2: 64 filters ===
    model.add(layers.Conv2D(64, (3,3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    
    model.add(layers.Conv2D(64, (3,3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    
    model.add(layers.MaxPooling2D((2,2)))  # 16×16 → 8×8
    model.add(layers.Dropout(0.25))
    
    # === Block 3: 128 filters ===
    model.add(layers.Conv2D(128, (3,3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    
    # Global Average Pooling instead of Flatten
    model.add(layers.GlobalAveragePooling2D())  # (8×8×128) → (128,)
    
    # === Dense Head ===
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))  # Higher dropout in dense head
    model.add(layers.Dense(10, activation='softmax'))
    
    return model

cnn_model = build_cnn()
cnn_model.summary()

In [ ]:
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=7,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-6, verbose=1
    )
]

print('Training CNN...')
cnn_history = cnn_model.fit(
    x_train, y_train,
    epochs=30,
    validation_split=0.1,
    batch_size=128,
    callbacks=cnn_callbacks,
    verbose=1
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test, y_test, verbose=0)
print(f'\n📊 CNN Results:')
print(f'   Test Accuracy : {cnn_test_acc:.4f} ({cnn_test_acc*100:.2f}%)')
print(f'   Test Loss     : {cnn_test_loss:.4f}')

plot_training_curves(cnn_history, 'CNN (VGG-Style)', '#8e44ad', '#e67e22')

---
# Section 7 — CNN + Data Augmentation

## Why Augmentation Matters — The Theory

With 50K training images and millions of possible pixel configurations per class, a model can memorize training examples rather than learning the concept. Augmentation breaks this by:

1. **Horizontal Flip**: A dog facing left = same dog facing right. Model learns flip invariance.
2. **Random Rotation (±15°)**: Objects aren't always upright. Rotation robustness.
3. **Zoom (±10%)**: Objects appear at varying distances from camera.
4. **Width/Height Shift**: Objects aren't always centered in frame.

**What we DO NOT use on CIFAR-10:**
- Vertical flip: A flipped car or airplane changes its class semantics
- Heavy color jitter: CIFAR-10 has meaningful color info (airplane sky vs frog grass)

**Implementation note:** TF's `tf.keras.Sequential` augmentation layers only apply during training (not inference) — no code needed to switch them off.

## Advanced: CutMix / MixUp 
Beyond geometric transforms, state-of-the-art uses:
- **MixUp**: Blend two images and their labels linearly → forces smooth decision boundaries
- **CutMix**: Paste a patch from one image onto another → richer local feature training
These techniques give +1-3% improvement but are overkill for this study.

In [ ]:
# Visualize augmentation effects
augmentation_preview = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name='augmentation_preview')

sample_img = x_train[0:1]  # Take first image, keep batch dim

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Data Augmentation — Same Image, 10 Random Variants', fontweight='bold')

for i, ax in enumerate(axes.flat):
    aug_img = augmentation_preview(sample_img, training=True)[0].numpy()
    ax.imshow(np.clip(aug_img, 0, 1))
    ax.axis('off')
    ax.set_title(f'Variant {i+1}', fontsize=9)

plt.tight_layout()
plt.show()
print('💡 Each variant is a valid training sample — same label, different view')

In [ ]:
def build_cnn_augmented():
    model = models.Sequential(name='CNN_Augmented')
    
    # Augmentation block — only active during training
    model.add(layers.RandomFlip('horizontal', input_shape=(32,32,3)))
    model.add(layers.RandomRotation(0.15))
    model.add(layers.RandomZoom(0.1))
    model.add(layers.RandomTranslation(0.1, 0.1))
    
    # Block 1
    model.add(layers.Conv2D(32, (3,3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(32, (3,3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))
    
    # Block 2
    model.add(layers.Conv2D(64, (3,3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(64, (3,3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))
    
    # Block 3
    model.add(layers.Conv2D(128, (3,3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(128, (3,3), padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.GlobalAveragePooling2D())
    
    # Dense head
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(10, activation='softmax'))
    
    return model

aug_cnn_model = build_cnn_augmented()
aug_cnn_model.summary()

In [ ]:
aug_cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Cosine decay: smooth LR schedule that prevents sharp oscillations
# Often outperforms step-based ReduceLROnPlateau for augmented training
lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=1e-3,
    first_decay_steps=10,
    t_mul=2.0,
    m_mul=0.9
)

aug_cnn_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=10,  # More patience — augmented models train slower
        restore_best_weights=True, verbose=1
    )
]

print('Training CNN + Augmentation...')
aug_history = aug_cnn_model.fit(
    x_train, y_train,
    epochs=40,  # Augmented models need more epochs
    validation_split=0.1,
    batch_size=128,
    callbacks=aug_cnn_callbacks,
    verbose=1
)

In [ ]:
aug_test_loss, aug_test_acc = aug_cnn_model.evaluate(x_test, y_test, verbose=0)
print(f'\n📊 CNN + Augmentation Results:')
print(f'   Test Accuracy : {aug_test_acc:.4f} ({aug_test_acc*100:.2f}%)')
print(f'   Test Loss     : {aug_test_loss:.4f}')

plot_training_curves(aug_history, 'CNN + Data Augmentation', '#27ae60', '#c0392b')

---
#  Section 8 — Comprehensive Performance Analysis

## Model Comparison Framework
When comparing ML models in practice, accuracy alone is insufficient. We also examine:
- **Generalization gap** (train vs test delta)
- **Parameter efficiency** (accuracy per million parameters)
- **Class-level performance** (which classes are hard?)
- **Confidence calibration** (does the model know when it's wrong?)

In [ ]:
# Collect predictions from all models
ann_preds_proba = ann_model.predict(x_test_flat, verbose=0)
cnn_preds_proba = cnn_model.predict(x_test, verbose=0)
aug_preds_proba = aug_cnn_model.predict(x_test, verbose=0)

ann_preds = np.argmax(ann_preds_proba, axis=1)
cnn_preds = np.argmax(cnn_preds_proba, axis=1)
aug_preds = np.argmax(aug_preds_proba, axis=1)

y_true = y_test.flatten()

# Build comparison dataframe
ann_params = ann_model.count_params()
cnn_params = cnn_model.count_params()
aug_params = aug_cnn_model.count_params()

results = pd.DataFrame({
    'Model': ['ANN Baseline', 'CNN (VGG-Style)', 'CNN + Augmentation'],
    'Test Accuracy (%)': [
        round(ann_test_acc * 100, 2),
        round(cnn_test_acc * 100, 2),
        round(aug_test_acc * 100, 2)
    ],
    'Test Loss': [
        round(ann_test_loss, 4),
        round(cnn_test_loss, 4),
        round(aug_test_loss, 4)
    ],
    'Parameters': [ann_params, cnn_params, aug_params],
    'Acc/M Params': [
        round(ann_test_acc / (ann_params / 1e6), 2),
        round(cnn_test_acc / (cnn_params / 1e6), 2),
        round(aug_test_acc / (aug_params / 1e6), 2)
    ]
})

print('=' * 70)
print('                  MODEL COMPARISON REPORT')
print('=' * 70)
print(results.to_string(index=False))
print('=' * 70)

In [ ]:
# Visual comparison of all three models
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('ANN vs CNN vs CNN+Aug — Validation Accuracy Over Training', 
             fontsize=13, fontweight='bold')

models_data = [
    (ann_history, 'ANN Baseline', '#e74c3c'),
    (cnn_history, 'CNN VGG-Style', '#3498db'),
    (aug_history, 'CNN + Augmentation', '#27ae60')
]

for ax, (hist, name, color) in zip(axes, models_data):
    epochs = range(1, len(hist.history['accuracy']) + 1)
    ax.plot(epochs, hist.history['accuracy'], color=color, 
            linewidth=2, label='Train')
    ax.plot(epochs, hist.history['val_accuracy'], color=color, 
            linewidth=2, linestyle='--', label='Val')
    best_val = max(hist.history['val_accuracy'])
    ax.set_title(f'{name}\nBest Val: {best_val:.3f}', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

---
#  Section 9 — Error Analysis

## The Confusion Matrix — Reading It Like a Pro

A confusion matrix `C[i][j]` = number of samples where true class is `i` but predicted as `j`.

- **Diagonal** = correct predictions
- **Off-diagonal** = specific failure modes
- **Asymmetric off-diagonals** = biased confusion (model confuses A with B more than B with A)

CIFAR-10 hard pairs to look for:
- Cat ↔ Dog (texture similarity)
- Automobile ↔ Truck (shape overlap)
- Bird ↔ Airplane (silhouette in sky)

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name, figsize=(10, 8)):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]  # Normalize rows
    
    fig, ax = plt.subplots(figsize=figsize)
    
    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', 
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        cmap='Blues', ax=ax, linewidths=0.5,
        vmin=0, vmax=1
    )
    
    ax.set_title(f'{model_name} — Normalized Confusion Matrix\n(Row: True, Col: Predicted)', 
                 fontweight='bold', fontsize=12)
    ax.set_xlabel('Predicted Class', fontweight='bold')
    ax.set_ylabel('True Class', fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    # Per-class accuracy
    per_class_acc = cm.diagonal() / cm.sum(axis=1)
    print(f'\n Per-class Accuracy ({model_name}):')
    for cls, acc in zip(CLASS_NAMES, per_class_acc):
        bar = '█' * int(acc * 20) + '░' * (20 - int(acc * 20))
        print(f'  {cls:<15}: {bar} {acc:.3f}')

# Best model: CNN with augmentation
plot_confusion_matrix(y_true, aug_preds, 'CNN + Augmentation')

In [ ]:
# Classification report — precision, recall, F1 for each class
print('=== Classification Report: CNN + Augmentation ===')
print(classification_report(y_true, aug_preds, target_names=CLASS_NAMES))

In [ ]:
# Visualize failure cases — where did our best model go wrong?
def show_failures(y_true, y_pred, proba, x_images, n=15, title='Failure Cases'):
    """Show misclassified examples with confidence scores."""
    wrong_idx = np.where(y_true != y_pred)[0]
    np.random.shuffle(wrong_idx)
    wrong_idx = wrong_idx[:n]
    
    rows = 3
    cols = 5
    fig, axes = plt.subplots(rows, cols, figsize=(16, 10))
    fig.suptitle(f'{title}: Misclassified Examples\n(True → Predicted, confidence)', 
                 fontweight='bold', fontsize=12)
    
    for ax, idx in zip(axes.flat, wrong_idx):
        ax.imshow(x_images[idx])
        conf = proba[idx][y_pred[idx]]
        true_cls = CLASS_NAMES[y_true[idx]]
        pred_cls = CLASS_NAMES[y_pred[idx]]
        ax.set_title(f'True: {true_cls}\nPred: {pred_cls} ({conf:.2f})', 
                     fontsize=8, color='red' if conf > 0.8 else 'orange')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    print('💡 Red titles = high-confidence wrong predictions (overconfident failures)')
    print('   Orange = uncertain predictions gone wrong (expected failure mode)')

show_failures(y_true, aug_preds, aug_preds_proba, x_test, title='CNN + Aug')

In [ ]:
# ANN vs CNN: Where does CNN win that ANN fails?
ann_wrong = set(np.where(y_true != ann_preds)[0])
cnn_right = set(np.where(y_true == aug_preds)[0])
cnn_recovered = ann_wrong & cnn_right

print(f'\n📊 Error Recovery Analysis:')
print(f'  ANN misclassified      : {len(ann_wrong):,} samples')
print(f'  CNN+Aug recovered      : {len(cnn_recovered):,} of those ({len(cnn_recovered)/len(ann_wrong)*100:.1f}%)')
print(f'  Remaining failures     : {len(ann_wrong) - len(cnn_recovered):,}')

# Which classes did CNN recover most?
recovered_classes = [CLASS_NAMES[y_true[i]] for i in cnn_recovered]
from collections import Counter
recovery_by_class = Counter(recovered_classes)

print(f'\n  Recovery by class (top 5):')
for cls, count in recovery_by_class.most_common(5):
    print(f'    {cls:<15}: {count:,} samples recovered')

---
#  Section 10 — Training Dynamics Deep Dive

## Understanding Your Training Curves Like a Pro

Reading training curves isn't just about watching accuracy go up. Key things to watch:

| Pattern | What it Means | What to Do |
|---|---|---|
| Train acc >> Val acc | Overfitting | More dropout, aug, weight decay |
| Both flat early | Underfitting | Bigger model, lower LR, more epochs |
| Val loss increases | Overfitting onset | EarlyStopping caught it |
| Spiky val loss | LR too high | ReduceLROnPlateau helps |
| Train == Val acc | Well generalized | Trust the test score |
| LR drop → accuracy jump | LR was the bottleneck | Fine-tune LR schedule |

In [ ]:
# Side-by-side learning curves — all models
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Complete Training Dynamics: ANN vs CNN vs CNN+Aug', 
             fontsize=14, fontweight='bold')

histories = [ann_history, cnn_history, aug_history]
model_names = ['ANN Baseline', 'CNN VGG-Style', 'CNN + Augmentation']
colors = ['#e74c3c', '#3498db', '#27ae60']

for col, (hist, name, color) in enumerate(zip(histories, model_names, colors)):
    epochs = range(1, len(hist.history['accuracy']) + 1)
    
    # Accuracy subplot
    ax_acc = axes[0][col]
    ax_acc.plot(epochs, hist.history['accuracy'], color=color, lw=2, label='Train')
    ax_acc.plot(epochs, hist.history['val_accuracy'], color=color, lw=2, 
                linestyle='--', alpha=0.7, label='Val')
    ax_acc.set_title(f'{name}\nAccuracy', fontweight='bold')
    ax_acc.set_ylim(0.3, 1.0)
    ax_acc.legend(fontsize=8)
    ax_acc.grid(True, alpha=0.4)
    ax_acc.set_ylabel('Accuracy')
    
    # Loss subplot
    ax_loss = axes[1][col]
    ax_loss.plot(epochs, hist.history['loss'], color=color, lw=2, label='Train')
    ax_loss.plot(epochs, hist.history['val_loss'], color=color, lw=2,
                 linestyle='--', alpha=0.7, label='Val')
    ax_loss.set_title(f'{name}\nLoss', fontweight='bold')
    ax_loss.legend(fontsize=8)
    ax_loss.grid(True, alpha=0.4)
    ax_loss.set_ylabel('Loss')
    ax_loss.set_xlabel('Epoch')

plt.tight_layout()
plt.show()

---
#  Section 11 — Final Comparison & Radar Chart

In [ ]:
# Final bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Final Performance Comparison', fontsize=13, fontweight='bold')

model_labels = ['ANN\nBaseline', 'CNN\nVGG-Style', 'CNN +\nAugmentation']
colors = ['#e74c3c', '#3498db', '#27ae60']

# Accuracy
accs = [ann_test_acc, cnn_test_acc, aug_test_acc]
bars = axes[0].bar(model_labels, [a*100 for a in accs], color=colors, 
                   edgecolor='black', linewidth=1.2)
axes[0].set_title('Test Accuracy (%)', fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].set_ylabel('Accuracy (%)')
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{acc*100:.1f}%', ha='center', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Loss
losses = [ann_test_loss, cnn_test_loss, aug_test_loss]
bars = axes[1].bar(model_labels, losses, color=colors, edgecolor='black', linewidth=1.2)
axes[1].set_title('Test Loss', fontweight='bold')
axes[1].set_ylabel('Loss')
for bar, loss in zip(bars, losses):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{loss:.3f}', ha='center', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Parameter efficiency
params_M = [ann_params/1e6, cnn_params/1e6, aug_params/1e6]
eff = [a/p for a, p in zip(accs, params_M)]
bars = axes[2].bar(model_labels, eff, color=colors, edgecolor='black', linewidth=1.2)
axes[2].set_title('Accuracy per Million Parameters', fontweight='bold')
axes[2].set_ylabel('Acc / M params')
for bar, e in zip(bars, eff):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{e:.2f}', ha='center', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Per-class F1 comparison — ANN vs Best CNN
from sklearn.metrics import f1_score

ann_f1  = f1_score(y_true, ann_preds, average=None)
aug_f1  = f1_score(y_true, aug_preds, average=None)

x = np.arange(len(CLASS_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, ann_f1, width, label='ANN', color='#e74c3c', alpha=0.8, edgecolor='black')
ax.bar(x + width/2, aug_f1, width, label='CNN + Aug', color='#27ae60', alpha=0.8, edgecolor='black')

ax.set_xlabel('Class')
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 Score: ANN vs CNN + Augmentation', fontweight='bold', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0.7, color='gray', linestyle=':', linewidth=1, label='0.7 threshold')

plt.tight_layout()
plt.show()

# Print delta
print('\nF1 Score Delta (CNN+Aug - ANN):')
for cls, ann_s, aug_s in zip(CLASS_NAMES, ann_f1, aug_f1):
    delta = aug_s - ann_s
    arrow = '▲' if delta > 0 else '▼'
    print(f'  {cls:<15}: ANN={ann_s:.3f}  CNN+Aug={aug_s:.3f}  {arrow} {abs(delta):.3f}')

---
#  Section 12 — Key Takeaways & Interview Insights

##  Core Findings

| Aspect | ANN | CNN | CNN + Aug |
|---|---|---|---|
| Architecture | Fully connected | Convolutional | Conv + Augmentation |
| Spatial awareness |  None |  Yes |  Yes |
| Weight sharing | No | Yes |  Yes |
| Parameter efficiency |  Low |  High |  High |
| Generalization | Poor | Good |  Best |
| Training data need |  More |  Moderate |  Less (effectively) |

---

##  Production-Level Lessons

### 1. Architecture Matters More Than Hyperparameters
The jump from ANN to CNN is much larger than any hyperparameter tuning on the ANN. Choosing the right inductive bias (conv for images, attention for sequences) is the highest-leverage decision.

### 2. Regularization Is Not Optional
On CIFAR-10 with 50K samples, a raw CNN without Dropout + BatchNorm will overfit within 5-10 epochs. Always regularize.

### 3. Data Augmentation Is Almost Free Performance
Augmentation costs zero additional data collection, zero annotation, and gives +3-5% accuracy. It should be the first thing you add after a working baseline.

### 4. Early Stopping + ReduceLROnPlateau = Robust Training
These two callbacks together save hours of GPU time and prevent overfitting from running too long. Always use them in production training.

### 5. Confusion Matrix > Aggregate Accuracy
A 75% accurate model that fails 95% of the time on one class is different from a 75% model with uniform failures. Always do per-class analysis, especially in production with business implications per class.

### 6. Parameter Count ≠ Model Quality
Our CNN has ~6x fewer parameters than ANN but significantly higher accuracy. Inductive bias (spatial locality assumption in CNN) is far more valuable than raw model capacity.

---

##  Next Steps to Push Further

| Technique | Expected Gain | Effort |
|---|---|---|
| Add L2 weight decay (1e-4) | +0.5-1% | Low |
| Label smoothing (0.1) | +0.5-1% | Low |
| LR warmup + cosine decay | +1-2% | Medium |
| CutMix / MixUp augmentation | +1-3% | Medium |
| ResNet-style skip connections | +3-5% | Medium |
| Transfer learning (EfficientNet) | +10-15% | Low (pretrained) |




In [ ]:
# Final summary print
print('=' * 60)
print('          FINAL EXPERIMENT SUMMARY')
print('=' * 60)
print(f'  ANN Baseline      : {ann_test_acc*100:.2f}% accuracy | {ann_params:,} params')
print(f'  CNN VGG-Style     : {cnn_test_acc*100:.2f}% accuracy | {cnn_params:,} params')
print(f'  CNN + Augmentation: {aug_test_acc*100:.2f}% accuracy | {aug_params:,} params')
print('=' * 60)
print(f'\n  CNN gain over ANN      : +{(cnn_test_acc - ann_test_acc)*100:.1f}%')
print(f'  Augmentation gain      : +{(aug_test_acc - cnn_test_acc)*100:.1f}%')
print(f'  Total CNN+Aug gain     : +{(aug_test_acc - ann_test_acc)*100:.1f}% over ANN')
print('=' * 60)